# CFPB Consumer Complaint Classifier — Complete Pipeline

I built this project to automatically classify consumer financial complaints into product categories using the CFPB (Consumer Financial Protection Bureau) database.

**What I cover in this notebook, end-to-end:**
1. **Data Acquisition** — Downloading and sampling 200k complaints from the raw ~2 GB CSV
2. **Preprocessing** — Cleaning text, consolidating product labels, splitting into train/val/test
3. **Feature Engineering** — TF-IDF sparse vectors + structured metadata features
4. **XGBoost Baseline** — Classical ML model as a performance benchmark
5. **SBERT Hybrid Model** — Semantic sentence embeddings to significantly boost minority-class performance
6. **Final Evaluation** — Honest results on a completely held-out test set

**Dataset:** CFPB Consumer Complaint Database (~7M complaints, ~2 GB raw CSV)
**Task:** Multi-class text classification into 7 financial product categories
**Best model:** SBERT (`all-MiniLM-L6-v2`) + XGBoost hybrid

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, html, zipfile, urllib.request
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
import xgboost as xgb

PROJECT_DIR = '/content/drive/MyDrive/nlp_project'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
FEATURE_DIR = os.path.join(PROJECT_DIR, 'features')
EMBED_DIR   = os.path.join(PROJECT_DIR, 'embeddings')
MODEL_DIR   = os.path.join(PROJECT_DIR, 'models')
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'outputs')
for d in [DATA_DIR, FEATURE_DIR, EMBED_DIR, MODEL_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

try:
    import subprocess
    subprocess.check_output('nvidia-smi', shell=True)
    DEVICE = 'cuda'
    print('GPU detected — using cuda')
except Exception:
    DEVICE = 'cpu'
    print('No GPU found — using cpu (slower)')

print('Setup complete.')
print(f'  Project : {PROJECT_DIR}')
print(f'  Data    : {DATA_DIR}')
print(f'  Models  : {MODEL_DIR}')

---
## Part 1 — Data Acquisition & First Look

I start by downloading the full CFPB Consumer Complaint Database. The raw CSV is about 500 MB zipped and ~2 GB unzipped, with roughly 7 million rows.

Loading the whole thing at once would crash the Colab session (free tier has a 12 GB RAM limit), so I read it in 50k-row chunks, filter immediately to only the rows that actually have a complaint narrative, and discard the rest. This keeps the in-memory footprint manageable — the full CSV is never loaded simultaneously.

At the end I save a stratified 200k sample to Drive. Everything else in this notebook builds on that file.

In [ ]:
NARRATIVE_COL = 'consumer_complaint_narrative'

DATA_URL = 'https://files.consumerfinance.gov/ccdb/complaints.csv.zip'
ZIP_PATH = '/content/complaints.csv.zip'
CSV_PATH = '/content/complaints.csv'

if not os.path.exists(CSV_PATH):
    print('Downloading (~500 MB) ...')
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print('Unzipping ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Done.')
else:
    print('CSV already on disk, skipping download.')

print(f'Raw CSV size: {os.path.getsize(CSV_PATH)/1e6:.1f} MB')

### Loading in chunks to avoid RAM crashes

I read 50,000 rows at a time. For each chunk I normalise column names and drop any row that has no narrative text. Only the filtered rows are kept in memory — this takes about 4–6 minutes but keeps RAM usage safe.

In [ ]:
CHUNK_SIZE = 50_000
chunks     = []
total_rows = 0
col_names  = None

print('Reading CSV in 50k-row chunks ...')
for i, chunk in enumerate(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, low_memory=False)):
    chunk.columns = (
        chunk.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace('-', '_', regex=False)
    )
    if col_names is None:
        col_names = chunk.columns.tolist()
    total_rows += len(chunk)
    if NARRATIVE_COL in chunk.columns:
        mask = chunk[NARRATIVE_COL].notna() & (chunk[NARRATIVE_COL].astype(str).str.strip() != '')
        if mask.any():
            chunks.append(chunk[mask])
    if (i + 1) % 30 == 0:
        kept = sum(len(c) for c in chunks)
        print(f'  Chunk {i+1:>4} | Processed: {total_rows:>8,} | Kept: {kept:>7,}')

df_text   = pd.concat(chunks, ignore_index=True)
n_with    = len(df_text)
n_without = total_rows - n_with

print(f'\nTotal rows in raw CSV    : {total_rows:,}')
print(f'Rows WITH narrative      : {n_with:,}  ({n_with/total_rows*100:.1f}%)')
print(f'Rows WITHOUT narrative   : {n_without:,}  ({n_without/total_rows*100:.1f}%)')
print(f'RAM usage                : {df_text.memory_usage(deep=True).sum()/1e6:.0f} MB')

### Checking which columns are present

In [ ]:
KEY_COLS = [
    NARRATIVE_COL, 'product', 'sub_product',
    'issue', 'sub_issue', 'company', 'state',
    'tags', 'submitted_via', 'date_received', 'complaint_id'
]
present = [c for c in KEY_COLS if c in df_text.columns]
missing = [c for c in KEY_COLS if c not in df_text.columns]

print(f'Present ({len(present)}): {present}')
print(f'Missing ({len(missing)}): {missing}')
print('\n--- Sample rows ---')
print(df_text[present].head(3).to_string())

### Product class distribution

I check how many unique product categories exist and how complaints are distributed across them. Heavy class imbalance directly affects model training, so I need to understand this early.

In [ ]:
prod_counts = df_text['product'].value_counts()
print(f'Unique Product classes   : {prod_counts.shape[0]}')
print(prod_counts.to_string())
print(f'\nImbalance ratio (max/min): {prod_counts.max()/prod_counts.min():.1f}x')

fig, ax = plt.subplots(figsize=(10, 5))
prod_counts.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('CFPB — Product Distribution (narrative-only records)', fontsize=13)
ax.set_xlabel('Number of Complaints')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'product_distribution.png'), dpi=120)
plt.show()

### Narrative length distribution

I check how long the complaint texts are. This tells me how much text I'm working with — and later helps me decide BERT's max token limit and truncation strategy.

In [ ]:
df_text['narrative_len'] = df_text[NARRATIVE_COL].str.count(' ') + 1

print('--- Narrative word count stats ---')
print(df_text['narrative_len'].describe(percentiles=[.25, .5, .75, .90, .95, .99]).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
df_text['narrative_len'].clip(upper=600).hist(bins=80, ax=ax, color='coral', edgecolor='white')
ax.set_title('Complaint Narrative Length (words, clipped at 600)', fontsize=12)
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'narrative_length_dist.png'), dpi=120)
plt.show()

### Saving a stratified 200k sample

I take a 200k sample stratified by product so every class is proportionally represented. This file is what all downstream parts load from — I never need to re-download or re-process the raw CSV.

In [ ]:
SAMPLE_SIZE = 200_000
df_text = df_text.dropna(subset=['product'])
frac = min(SAMPLE_SIZE / len(df_text), 1.0)

df_sample = (
    df_text
    .groupby('product', group_keys=False)
    .apply(lambda x: x.sample(frac=frac, random_state=42))
    .reset_index(drop=True)
)
print(f'Sample size: {len(df_sample):,}')
print('\nProduct distribution in sample:')
print(df_sample['product'].value_counts().to_string())

In [ ]:
KEEP_COLS = [
    'complaint_id', 'date_received', 'product', 'sub_product',
    'issue', 'sub_issue', NARRATIVE_COL,
    'company', 'state', 'tags', 'submitted_via', 'narrative_len'
]
keep     = [c for c in KEEP_COLS if c in df_sample.columns]
df_final = df_sample[keep].copy()

OUT_PATH = os.path.join(DATA_DIR, 'complaints_sample.csv')
df_final.to_csv(OUT_PATH, index=False)
print(f'Saved  : {OUT_PATH}')
print(f'Rows   : {len(df_final):,}')
print(f'Size   : {os.path.getsize(OUT_PATH)/1e6:.1f} MB')
print(f'Columns: {df_final.columns.tolist()}')

---
## Part 2 — Preprocessing & Label Consolidation

The product labels in the raw data are messy. The CFPB renamed its product taxonomy twice — once in 2017 and once in 2020 — leaving 21 distinct label strings in the historical data that really represent only 8 unique product categories. If I train on the raw labels, the model wastes capacity learning the difference between names for the same product rather than learning about actual complaint content.

I fix this by mapping all 21 raw labels to 8 canonical classes. Then I clean the complaint text (removing redaction markers, HTML entities, excess whitespace), encode labels as integers, and split into train/val/test sets.

The test set gets locked away after this point. I won't look at it again until Part 6.

In [ ]:
LABEL_COL = 'product_clean'
IN_PATH   = os.path.join(DATA_DIR, 'complaints_sample.csv')
df        = pd.read_csv(IN_PATH, low_memory=False)

print(f'Loaded: {len(df):,} rows x {df.shape[1]} columns')
print('\nRaw product value counts:')
print(df['product'].value_counts().to_string())

### Consolidating product labels

The CFPB renamed its taxonomy twice, so what looks like 21 classes is really just 8. Without this step the model would learn naming conventions instead of complaint content patterns.

My mapping groups all credit reporting variants together, all loan types together, and so on.

In [ ]:
PRODUCT_MAP = {
    # Three era-names for the same credit reporting category
    'Credit reporting':                                                             'Credit Reporting',
    'Credit reporting or other personal consumer reports':                          'Credit Reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',

    'Debt collection':                                                              'Debt Collection',

    # Credit card variants including prepaid
    'Credit card':                                                                  'Credit Card',
    'Credit card or prepaid card':                                                  'Credit Card',
    'Prepaid card':                                                                 'Credit Card',

    # Banking
    'Bank account or service':                                                      'Banking',
    'Checking or savings account':                                                  'Banking',

    'Mortgage':                                                                     'Mortgage',

    # All loan types under one umbrella
    'Student loan':                                                                 'Loans',
    'Vehicle loan or lease':                                                        'Loans',
    'Consumer Loan':                                                                'Loans',
    'Payday loan':                                                                  'Loans',
    'Payday loan, title loan, or personal loan':                                    'Loans',
    'Payday loan, title loan, personal loan, or advance loan':                      'Loans',

    # Money transfer variants
    'Money transfers':                                                              'Money Transfer',
    'Money transfer, virtual currency, or money service':                           'Money Transfer',
    'Virtual currency':                                                             'Money Transfer',

    # Too few records to model as a standalone class
    'Debt or credit management':                                                    'Other',
    'Other financial service':                                                      'Other',
}

df[LABEL_COL] = df['product'].map(PRODUCT_MAP)

unmapped = df[df[LABEL_COL].isna()]['product'].unique()
if len(unmapped):
    print(f'WARNING — unmapped product labels: {unmapped}')
else:
    print('All product labels mapped successfully.')

print('\nConsolidated class distribution:')
counts = df[LABEL_COL].value_counts()
print(counts.to_string())
print(f'\nTotal classes        : {counts.shape[0]}')
print(f'Imbalance ratio      : {counts.max()/counts.min():.1f}x')

In [ ]:
# Drop the tiny 'Other' class — only ~282 records, not enough to learn from
before = len(df)
df = df[df[LABEL_COL] != 'Other'].copy()
print(f'Dropped {before - len(df)} rows labelled Other. Remaining: {len(df):,}')

print('\nFinal class counts:')
print(df[LABEL_COL].value_counts().to_string())

fig, ax = plt.subplots(figsize=(9, 4))
df[LABEL_COL].value_counts().sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Consolidated Product Classes', fontsize=13)
ax.set_xlabel('Count')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'consolidated_classes.png'), dpi=120)
plt.show()

### Cleaning the complaint text

The CFPB narrative field has some specific noise I need to strip out before feeding it to any model:

| Pattern | Why it's noise | Fix |
|---|---|---|
| `XXXX`, `XX/XX/XXXX` | CFPB redacts PII with X's — these add no meaning | Remove |
| `&amp;`, `&lt;` | HTML entities from copy-paste | Unescape |
| `\n`, `\r`, `\t` | Newlines and tabs | Replace with space |
| Multiple spaces | Residue after above removals | Collapse to one space |

I intentionally keep the original capitalisation — BERT (used in Part 5) benefits from proper-noun casing signals.

In [ ]:
def clean_narrative(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = html.unescape(text)
    text = re.sub(r'X{2,}(?:/X{2,})*', '', text)
    text = re.sub(r'[\n\r\t]', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

df['narrative_clean'] = df[NARRATIVE_COL].apply(clean_narrative)

print('--- Before / After cleaning (3 examples) ---\n')
for idx in df.sample(3, random_state=1).index:
    print(f'RAW   : {df.loc[idx, NARRATIVE_COL][:200]}')
    print(f'CLEAN : {df.loc[idx, "narrative_clean"][:200]}')
    print()

In [ ]:
before = len(df)
df = df[df['narrative_clean'].str.len() >= 10].copy()
print(f'Dropped {before - len(df)} near-empty rows. Remaining: {len(df):,}')

df['narrative_len'] = df['narrative_clean'].str.count(' ') + 1
print('\nCleaned narrative word count stats:')
print(df['narrative_len'].describe(percentiles=[.5, .75, .95, .99]).to_string())

### Label encoding

I create a sorted, stable integer encoding for the class names and save it to Drive. Every part of this pipeline loads the same mapping file, so predictions are always interpretable and consistent across runs.

In [ ]:
classes  = sorted(df[LABEL_COL].unique())
label2id = {c: i for i, c in enumerate(classes)}
id2label = {i: c for c, i in label2id.items()}

df['label'] = df[LABEL_COL].map(label2id)

print('Label encoding:')
for name, idx in label2id.items():
    print(f'  {idx}  {name}')

mapping_path = os.path.join(DATA_DIR, 'label_mapping.json')
with open(mapping_path, 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=2)
print(f'\nSaved: {mapping_path}')

### Train / Val / Test split

I split stratified by label so every class appears proportionally in all three sets:

- **Train (70%)** — what the model learns from
- **Val (15%)** — hyperparameter tuning and early stopping decisions
- **Test (15%)** — completely locked away until Part 6

I'll never touch the test set again until the final evaluation.

In [ ]:
df_trainval, df_test = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=42
)
df_train, df_val = train_test_split(
    df_trainval, test_size=(0.15/0.85), stratify=df_trainval['label'], random_state=42
)

print(f'Train : {len(df_train):>7,} rows  ({len(df_train)/len(df)*100:.1f}%)')
print(f'Val   : {len(df_val):>7,} rows  ({len(df_val)/len(df)*100:.1f}%)')
print(f'Test  : {len(df_test):>7,} rows  ({len(df_test)/len(df)*100:.1f}%)')
print('\nClass distribution in train:')
print(df_train[LABEL_COL].value_counts().to_string())

### Class weights

Credit Reporting makes up about 66% of the data. Without correcting for this, a model can achieve high accuracy just by predicting Credit Reporting for almost everything. I compute balanced class weights so the model is penalised more for errors on minority classes.

In [ ]:
cw = compute_class_weight(
    class_weight='balanced',
    classes=np.array(sorted(label2id.values())),
    y=df_train['label'].values
)
class_weights = dict(enumerate(cw))

print('Class weights (higher = penalised more for missing this class):')
for idx, w in class_weights.items():
    print(f'  {idx}  {id2label[idx]:<45}  weight: {w:.3f}')

cw_path = os.path.join(DATA_DIR, 'class_weights.json')
with open(cw_path, 'w') as f:
    json.dump({str(k): v for k, v in class_weights.items()}, f, indent=2)
print(f'\nSaved: {cw_path}')

In [ ]:
SAVE_COLS = [
    'complaint_id', 'narrative_clean', 'label', LABEL_COL,
    'narrative_len', 'company', 'state', 'tags', 'submitted_via'
]
save_cols = [c for c in SAVE_COLS if c in df_train.columns]

for name, split_df in [('train', df_train), ('val', df_val), ('test', df_test)]:
    path = os.path.join(DATA_DIR, f'{name}.csv')
    split_df[save_cols].to_csv(path, index=False)
    print(f'{name:5s} → {path}  ({len(split_df):,} rows, {os.path.getsize(path)/1e6:.1f} MB)')

---
## Part 3 — Feature Engineering

I build two kinds of features that I'll combine into a single matrix:

1. **TF-IDF text features** — a sparse bag-of-words representation using unigrams and bigrams (30k vocabulary). This is the foundation for the XGBoost baseline.
2. **Metadata features** — structured signals from the submission channel, geographic region, complaint length, and demographic tags.

The critical rule: I fit all transformers **on the training set only**, then transform val and test separately. Fitting on all data first would leak test-set information into the model.

In [ ]:
TEXT_COL = 'narrative_clean'

df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

with open(os.path.join(DATA_DIR, 'label_mapping.json')) as f:
    mapping  = json.load(f)
    id2label = {int(k): v for k, v in mapping['id2label'].items()}
    label2id = mapping['label2id']

print(f'Train : {len(df_train):,}')
print(f'Val   : {len(df_val):,}')
print(f'Test  : {len(df_test):,}')

### TF-IDF vectorizer

My key choices:
- **Unigrams + bigrams** (`ngram_range=(1,2)`) — bigrams catch phrases like "credit card" vs just "credit"
- **30,000 vocabulary cap** — enough for this domain without sparse noise
- **`min_df=5`** — ignores terms appearing in fewer than 5 documents (typos, one-off jargon)
- **`sublinear_tf=True`** — uses log(1+tf) to dampen very high-frequency terms
- **English stopwords removed** — strips 'the', 'and', 'to', etc.

I fit on train only, then transform val and test separately.

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range  = (1, 2),
    max_features = 30_000,
    min_df       = 5,
    sublinear_tf = True,
    stop_words   = 'english',
    analyzer     = 'word',
    token_pattern= r'\b[a-zA-Z][a-zA-Z0-9]+\b',
)

print('Fitting TF-IDF on train ...')
X_tfidf_train = tfidf.fit_transform(df_train[TEXT_COL].fillna(''))

print('Transforming val and test ...')
X_tfidf_val  = tfidf.transform(df_val[TEXT_COL].fillna(''))
X_tfidf_test = tfidf.transform(df_test[TEXT_COL].fillna(''))

print(f'\nVocabulary size   : {len(tfidf.vocabulary_):,}')
print(f'Train shape       : {X_tfidf_train.shape}  (sparse)')
print(f'Sparsity (train)  : {1 - X_tfidf_train.nnz/(X_tfidf_train.shape[0]*X_tfidf_train.shape[1]):.3%}')

tfidf_path = os.path.join(FEATURE_DIR, 'tfidf_vectorizer.joblib')
joblib.dump(tfidf, tfidf_path)
print(f'Saved: {tfidf_path}')

### Sanity check — top TF-IDF terms per class

I compute the mean TF-IDF score across all training documents in each class. The top terms should be domain-specific: "mortgage" ranking first for Mortgage, "debt" ranking first for Debt Collection, etc. If that's what I see, the features are working correctly.

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
y_train       = df_train['label'].values

print('Top 10 TF-IDF terms per class:\n')
for label_id in sorted(id2label):
    mask       = (y_train == label_id)
    mean_tfidf = np.asarray(X_tfidf_train[mask].mean(axis=0)).flatten()
    top_idx    = mean_tfidf.argsort()[-10:][::-1]
    print(f'{id2label[label_id]}')
    print(f'  {list(feature_names[top_idx])}')
    print()

### Metadata features

Beyond the text, I have structured fields that carry real signal:

| Feature | Encoding | Why it's useful |
|---|---|---|
| `submitted_via` | One-hot | Submission channel (Web/Phone/Fax) correlates with product type |
| `tags` | Binary flags | "Older American" and "Servicemember" tags link to specific complaint types |
| `narrative_len` | StandardScaler | Longer complaints tend to be mortgage or loan disputes |
| `state_region` | One-hot (4 regions) | Geographic patterns in complaint types |

I skip raw state (51 values = too many sparse columns) and company name (thousands of unique values). I map states to 4 US census regions instead.

In [ ]:
REGION_MAP = {
    'CT':'Northeast','ME':'Northeast','MA':'Northeast','NH':'Northeast',
    'RI':'Northeast','VT':'Northeast','NJ':'Northeast','NY':'Northeast','PA':'Northeast',
    'IN':'Midwest','IL':'Midwest','MI':'Midwest','OH':'Midwest','WI':'Midwest',
    'IA':'Midwest','KS':'Midwest','MN':'Midwest','MO':'Midwest','NE':'Midwest',
    'ND':'Midwest','SD':'Midwest',
    'DE':'South','FL':'South','GA':'South','MD':'South','NC':'South','SC':'South',
    'VA':'South','DC':'South','WV':'South','AL':'South','KY':'South','MS':'South',
    'TN':'South','AR':'South','LA':'South','OK':'South','TX':'South',
    'AZ':'West','CO':'West','ID':'West','MT':'West','NV':'West','NM':'West',
    'UT':'West','WY':'West','AK':'West','CA':'West','HI':'West','OR':'West','WA':'West',
}

def build_metadata(df_in):
    tags = df_in['tags'].fillna('').astype(str)
    return pd.DataFrame({
        'submitted_via'     : df_in['submitted_via'].fillna('Unknown').astype(str),
        'state_region'      : df_in['state'].fillna('').astype(str).map(REGION_MAP).fillna('Unknown'),
        'is_older_american' : tags.str.contains('Older American', na=False).astype(int),
        'is_servicemember'  : tags.str.contains('Servicemember',  na=False).astype(int),
        'narrative_len'     : df_in['narrative_len'].fillna(0).astype(float),
    })

meta_train = build_metadata(df_train)
meta_val   = build_metadata(df_val)
meta_test  = build_metadata(df_test)
print(f'Metadata shape: {meta_train.shape}')
print(meta_train.head(3).to_string())

In [ ]:
CAT_COLS = ['submitted_via', 'state_region']
NUM_COLS = ['is_older_american', 'is_servicemember', 'narrative_len']

ohe    = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
scaler = StandardScaler()

cat_train = ohe.fit_transform(meta_train[CAT_COLS])
num_train = scaler.fit_transform(meta_train[NUM_COLS])

cat_val  = ohe.transform(meta_val[CAT_COLS])
num_val  = scaler.transform(meta_val[NUM_COLS])
cat_test = ohe.transform(meta_test[CAT_COLS])
num_test = scaler.transform(meta_test[NUM_COLS])

num_train_sp = sp.csr_matrix(num_train)
num_val_sp   = sp.csr_matrix(num_val)
num_test_sp  = sp.csr_matrix(num_test)

print(f'OHE categories: {ohe.get_feature_names_out().tolist()}')

joblib.dump({'ohe': ohe, 'scaler': scaler}, os.path.join(FEATURE_DIR, 'meta_encoder.joblib'))
print('Saved meta_encoder.joblib')

### Combining TF-IDF + metadata → final sparse matrix

I horizontally stack the TF-IDF matrix with the metadata matrix into one sparse matrix per split. Shape: `(n_rows, 30,000 TF-IDF + n_metadata_features)`.

In [ ]:
X_train = sp.hstack([X_tfidf_train, cat_train, num_train_sp], format='csr')
X_val   = sp.hstack([X_tfidf_val,   cat_val,   num_val_sp],   format='csr')
X_test  = sp.hstack([X_tfidf_test,  cat_test,  num_test_sp],  format='csr')

y_train = df_train['label'].values
y_val   = df_val['label'].values
y_test  = df_test['label'].values

print(f'X_train : {X_train.shape}  nnz={X_train.nnz:,}')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')

for name, mat in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test)]:
    path = os.path.join(FEATURE_DIR, f'{name}.npz')
    sp.save_npz(path, mat)
    print(f'Saved {name}.npz  ({os.path.getsize(path)/1e6:.1f} MB)')

for name, arr in [('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
    path = os.path.join(FEATURE_DIR, f'{name}.npy')
    np.save(path, arr)
    print(f'Saved {name}.npy')

---
## Part 4 — XGBoost Baseline Model

I train XGBoost on the TF-IDF + metadata feature matrix from Part 3. This is my **classical baseline** — it trains in about 2 minutes, is interpretable through feature importance, and gives me a performance floor that the semantic model in Part 5 needs to beat.

**A note on a training pitfall I hit:** combining `sample_weight` with early stopping caused training to stop after 0–2 iterations. The conflict is that early stopping watches unweighted validation loss, but sample weights shift the training objective, so the two metrics diverge immediately. My fix was to drop `sample_weight` and let early stopping work correctly on the unweighted validation loss. The result was stable training with proper convergence.

I run on GPU (T4) — with `device='cuda'` XGBoost is about 10× faster on this 140k × 30k sparse matrix.

In [ ]:
print('Loading feature matrices ...')
X_train = sp.load_npz(os.path.join(FEATURE_DIR, 'X_train.npz'))
X_val   = sp.load_npz(os.path.join(FEATURE_DIR, 'X_val.npz'))
X_test  = sp.load_npz(os.path.join(FEATURE_DIR, 'X_test.npz'))

y_train = np.load(os.path.join(FEATURE_DIR, 'y_train.npy'))
y_val   = np.load(os.path.join(FEATURE_DIR, 'y_val.npy'))
y_test  = np.load(os.path.join(FEATURE_DIR, 'y_test.npy'))

with open(os.path.join(DATA_DIR, 'label_mapping.json')) as f:
    mapping  = json.load(f)
    id2label = {int(k): v for k, v in mapping['id2label'].items()}

CLASS_NAMES = [id2label[i] for i in sorted(id2label)]
N_CLASSES   = len(CLASS_NAMES)
print(f'X_train : {X_train.shape}')
print(f'Classes : {CLASS_NAMES}')

### Training XGBoost

Key choices:
- `n_estimators=3000` with `early_stopping_rounds=100` — high ceiling, let early stopping find the optimal point
- `learning_rate=0.05` — moderate; needs ~400–800 trees to converge at this rate
- `colsample_bytree=0.5` — samples 15k of 30k features per tree for regularization
- No `sample_weight` — avoids the early-stopping conflict described above

In [ ]:
model_xgb = xgb.XGBClassifier(
    n_estimators          = 3000,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.5,
    min_child_weight      = 5,
    objective             = 'multi:softprob',
    num_class             = N_CLASSES,
    eval_metric           = 'mlogloss',
    device                = DEVICE,
    tree_method           = 'hist',
    random_state          = 42,
    verbosity             = 1,
    early_stopping_rounds = 100,
)

print('Training XGBoost (up to 3000 trees, patience=100) ...')
model_xgb.fit(
    X_train, y_train,
    eval_set = [(X_val, y_val)],
    verbose  = 200,
)
print(f'\nBest iteration : {model_xgb.best_iteration}')
print(f'Best val loss  : {model_xgb.best_score:.4f}')

### Validation results

I evaluate on **val** only here — the test set stays locked until Part 6. I use both accuracy and macro F1. Macro F1 weights every class equally, so poor performance on minority classes is penalised just as much as poor performance on Credit Reporting.

In [ ]:
y_pred_val   = model_xgb.predict(X_val)
acc_xgb      = accuracy_score(y_val, y_pred_val)
macro_f1_xgb = f1_score(y_val, y_pred_val, average='macro')

print(f'Validation Accuracy : {acc_xgb:.4f}  ({acc_xgb*100:.2f}%)')
print(f'Validation Macro F1 : {macro_f1_xgb:.4f}  ({macro_f1_xgb*100:.2f}%)')
print()
print('--- Per-class report ---')
print(classification_report(y_val, y_pred_val, target_names=CLASS_NAMES, digits=3))

### Confusion matrix

In [ ]:
cm      = confusion_matrix(y_val, y_pred_val)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual',    fontsize=12)
ax.set_title('XGBoost Baseline — Normalised Confusion Matrix (Val Set)', fontsize=13)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'xgb_confusion_matrix.png'), dpi=120)
plt.show()

### Feature importance

XGBoost's gain-based importance shows which features reduce loss the most across all splits they're used in. The top features should be domain-specific terms — this is a sanity check that the model is learning real signal and not noise.

In [ ]:
tfidf_vec = joblib.load(os.path.join(FEATURE_DIR, 'tfidf_vectorizer.joblib'))
meta_enc  = joblib.load(os.path.join(FEATURE_DIR, 'meta_encoder.joblib'))
ohe       = meta_enc['ohe']

all_names = np.array(
    list(tfidf_vec.get_feature_names_out()) +
    list(ohe.get_feature_names_out()) +
    ['is_older_american', 'is_servicemember', 'narrative_len']
)

scores     = model_xgb.get_booster().get_score(importance_type='gain')
importance = np.zeros(len(all_names))
for fname, score in scores.items():
    idx = int(fname[1:])
    if idx < len(all_names):
        importance[idx] = score

top_n   = 25
top_idx = importance.argsort()[-top_n:][::-1]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(all_names[top_idx][::-1], importance[top_idx][::-1], color='steelblue', edgecolor='white')
ax.set_title(f'XGBoost — Top {top_n} Features by Gain', fontsize=13)
ax.set_xlabel('Gain')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'xgb_feature_importance.png'), dpi=120)
plt.show()

print('Top 25 features:')
for i in top_idx[:25]:
    print(f'  {all_names[i]:<35}  gain: {importance[i]:.1f}')

In [ ]:
model_path = os.path.join(MODEL_DIR, 'xgb_baseline.json')
model_xgb.save_model(model_path)
print(f'Saved: {model_path}  ({os.path.getsize(model_path)/1e6:.1f} MB)')

---
## Part 5 — SBERT Embeddings + Hybrid XGBoost

TF-IDF treats every word as independent — it doesn't understand that "debt collection" and "debt collector" are related, or that "credit reporting agency" and "credit bureau" mean the same thing. This is why classes like Debt Collection get poor F1 scores from the baseline (F1 ~0.15).

I fix this by using **SBERT** (`all-MiniLM-L6-v2`) to encode each complaint into a 384-dimensional dense vector that captures semantic meaning rather than just word counts.

I train and compare three models:
1. **TF-IDF baseline** (from Part 4) — the reference point
2. **SBERT-only** — semantic embeddings alone, no TF-IDF
3. **Hybrid (TF-IDF + SBERT)** — both combined; I expect this to be the strongest

The big expected gain is on Debt Collection — where semantic similarity should push F1 from ~0.15 to 0.55+.

In [ ]:
!pip install -q sentence-transformers
print('sentence-transformers installed.')

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

with open(os.path.join(DATA_DIR, 'label_mapping.json')) as f:
    mapping  = json.load(f)
    id2label = {int(k): v for k, v in mapping['id2label'].items()}

CLASS_NAMES = [id2label[i] for i in sorted(id2label)]
N_CLASSES   = len(CLASS_NAMES)

y_train = df_train['label'].values
y_val   = df_val['label'].values
y_test  = df_test['label'].values

print(f'Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')

### Loading the SBERT model

I use `all-MiniLM-L6-v2`:
- 384-dimensional output embeddings
- 6 transformer layers (distilled from 12-layer BERT — much faster, minimal quality loss)
- Ranks top-5 on the MTEB sentence embedding benchmark
- ~80 MB, encodes ~400 sentences/sec on T4 GPU

I enable `normalize_embeddings=True` so each vector is L2-normalized to unit length. This makes cosine similarity equivalent to dot product and helps the downstream classifier.

In [ ]:
print('Loading all-MiniLM-L6-v2 ...')
sbert = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
print(f'Embedding dim  : {sbert.get_sentence_embedding_dimension()}')
print(f'Max seq length : {sbert.max_seq_length} tokens')
print(f'Device         : {DEVICE}')

### Encoding all three splits

I encode train, val, and test separately and cache them to Drive immediately. If the session crashes mid-encoding, I can reload without starting over.

Expected times on T4 GPU: train (~139k texts) takes 6–8 minutes; val and test (~30k each) take about 1–2 minutes each.

In [ ]:
def encode_and_save(model, df, split_name, embed_dir, text_col='narrative_clean', batch_size=256):
    save_path = os.path.join(embed_dir, f'sbert_{split_name}.npy')
    if os.path.exists(save_path):
        emb = np.load(save_path)
        print(f'{split_name}: loaded from cache  {emb.shape}')
        return emb
    texts = df[text_col].fillna('').tolist()
    print(f'Encoding {split_name} ({len(texts):,} texts) ...')
    emb = model.encode(
        texts,
        batch_size           = batch_size,
        show_progress_bar    = True,
        normalize_embeddings = True,
        convert_to_numpy     = True,
    )
    np.save(save_path, emb)
    print(f'  Saved: {save_path}  shape={emb.shape}  size={os.path.getsize(save_path)/1e6:.1f} MB')
    return emb

emb_train = encode_and_save(sbert, df_train, 'train', EMBED_DIR)
emb_val   = encode_and_save(sbert, df_val,   'val',   EMBED_DIR)
emb_test  = encode_and_save(sbert, df_test,  'test',  EMBED_DIR)

print(f'\nAll embeddings ready.')
print(f'  emb_train : {emb_train.shape}')
print(f'  emb_val   : {emb_val.shape}')

### Sanity check — are the embeddings meaningful?

Before training I pick one complaint per class from the val set and find its nearest neighbor. If SBERT is working correctly, complaints about the same product should cluster together, so the nearest neighbor should usually be from the same class.

In [ ]:
print('Nearest-neighbour sanity check (same class = good embedding)\n')
np.random.seed(42)
for label_id in sorted(id2label):
    idx_pool  = np.where(y_val == label_id)[0]
    query_idx = np.random.choice(idx_pool)
    query_emb = emb_val[query_idx:query_idx+1]

    sims = cosine_similarity(query_emb, emb_val)[0]
    sims[query_idx] = -1
    top3 = sims.argsort()[-3:][::-1]

    top3_classes = [id2label[y_val[i]] for i in top3]
    match = sum(1 for c in top3_classes if c == id2label[label_id])
    print(f'Query class : {id2label[label_id]}')
    print(f'Top-3 NN    : {top3_classes}  — {match}/3 same class')
    print()

### Building the hybrid feature matrix

I combine TF-IDF (sparse, 30,009 features) with SBERT embeddings (dense, 384 features) by horizontally stacking them. TF-IDF contributes keyword frequency signal; SBERT contributes semantic meaning. Together they're stronger than either alone.

In [ ]:
X_tfidf_train = sp.load_npz(os.path.join(FEATURE_DIR, 'X_train.npz'))
X_tfidf_val   = sp.load_npz(os.path.join(FEATURE_DIR, 'X_val.npz'))
X_tfidf_test  = sp.load_npz(os.path.join(FEATURE_DIR, 'X_test.npz'))

def build_hybrid(X_sparse, emb_dense):
    return sp.hstack(
        [X_sparse, sp.csr_matrix(emb_dense.astype(np.float32))],
        format='csr'
    )

X_hybrid_train = build_hybrid(X_tfidf_train, emb_train)
X_hybrid_val   = build_hybrid(X_tfidf_val,   emb_val)
X_hybrid_test  = build_hybrid(X_tfidf_test,  emb_test)

print(f'Hybrid matrix shapes:')
print(f'  Train : {X_hybrid_train.shape}  (30,009 TF-IDF + 384 SBERT = {X_hybrid_train.shape[1]})')
print(f'  Val   : {X_hybrid_val.shape}')

### XGBoost on SBERT-only features

In [ ]:
X_sbert_train = sp.csr_matrix(emb_train.astype(np.float32))
X_sbert_val   = sp.csr_matrix(emb_val.astype(np.float32))

model_sbert = xgb.XGBClassifier(
    n_estimators          = 3000,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 5,
    objective             = 'multi:softprob',
    num_class             = N_CLASSES,
    eval_metric           = 'mlogloss',
    device                = DEVICE,
    tree_method           = 'hist',
    random_state          = 42,
    early_stopping_rounds = 100,
)

print('Training XGBoost on SBERT-only features (384 dims) ...')
model_sbert.fit(X_sbert_train, y_train, eval_set=[(X_sbert_val, y_val)], verbose=200)

y_pred_sbert = model_sbert.predict(X_sbert_val)
acc_sbert    = accuracy_score(y_val, y_pred_sbert)
f1_sbert     = f1_score(y_val, y_pred_sbert, average='macro')
print(f'\n[SBERT-only]  Accuracy: {acc_sbert:.4f}  Macro F1: {f1_sbert:.4f}')
print(f'Best iteration: {model_sbert.best_iteration}')

### XGBoost on hybrid features (TF-IDF + SBERT)

`colsample_bytree=0.3` samples ~9k features per tree from the 30,393 total, ensuring both TF-IDF and SBERT features get selected in every tree.

In [ ]:
model_hybrid = xgb.XGBClassifier(
    n_estimators          = 3000,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.3,
    min_child_weight      = 5,
    objective             = 'multi:softprob',
    num_class             = N_CLASSES,
    eval_metric           = 'mlogloss',
    device                = DEVICE,
    tree_method           = 'hist',
    random_state          = 42,
    early_stopping_rounds = 100,
)

print('Training XGBoost on hybrid features (TF-IDF + SBERT) ...')
model_hybrid.fit(X_hybrid_train, y_train, eval_set=[(X_hybrid_val, y_val)], verbose=200)

y_pred_hybrid = model_hybrid.predict(X_hybrid_val)
acc_hybrid    = accuracy_score(y_val, y_pred_hybrid)
f1_hybrid     = f1_score(y_val, y_pred_hybrid, average='macro')
print(f'\n[Hybrid] Accuracy: {acc_hybrid:.4f}  Macro F1: {f1_hybrid:.4f}')
print(f'Best iteration: {model_hybrid.best_iteration}')

### Comparing all three models side by side

In [ ]:
f1_sbert_pc  = f1_score(y_val, y_pred_sbert,  average=None)
f1_hybrid_pc = f1_score(y_val, y_pred_hybrid, average=None)
tfidf_f1_ref = [0.382, 0.457, 0.851, 0.150, 0.369, 0.345, 0.343]

print('=' * 70)
print(f'{"Class":<28} {"TF-IDF":>10} {"SBERT":>10} {"Hybrid":>10}')
print('-' * 70)
for i, name in enumerate(CLASS_NAMES):
    print(f'{name:<28} {tfidf_f1_ref[i]:>10.3f} {f1_sbert_pc[i]:>10.3f} {f1_hybrid_pc[i]:>10.3f}')
print('-' * 70)
print(f'{"Macro F1":<28} {0.414:>10.3f} {f1_sbert:>10.3f} {f1_hybrid:>10.3f}')
print(f'{"Accuracy":<28} {0.7118:>10.3f} {acc_sbert:>10.3f} {acc_hybrid:>10.3f}')
print('=' * 70)

x = np.arange(len(CLASS_NAMES))
w = 0.26
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w, tfidf_f1_ref,  w, label='TF-IDF',       color='steelblue', edgecolor='white')
ax.bar(x,     f1_sbert_pc,   w, label='SBERT-only',    color='coral',     edgecolor='white')
ax.bar(x + w, f1_hybrid_pc,  w, label='Hybrid (best)', color='seagreen',  edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=25, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1: TF-IDF vs SBERT vs Hybrid (Validation Set)', fontsize=13)
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=120)
plt.show()

In [ ]:
cm      = confusion_matrix(y_val, y_pred_hybrid)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual',    fontsize=12)
ax.set_title('Hybrid Model (TF-IDF + SBERT) — Normalised Confusion Matrix', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'hybrid_confusion_matrix.png'), dpi=120)
plt.show()

In [ ]:
best_model = model_hybrid if f1_hybrid >= f1_sbert else model_sbert
best_name  = 'hybrid' if f1_hybrid >= f1_sbert else 'sbert_only'

path = os.path.join(MODEL_DIR, f'xgb_{best_name}.json')
best_model.save_model(path)
print(f'Best model  : {best_name}')
print(f'Saved to    : {path}')
print(f'Size        : {os.path.getsize(path)/1e6:.1f} MB')

In [ ]:
print('Full classification report — Hybrid model (val set):')
print(classification_report(y_val, y_pred_hybrid, target_names=CLASS_NAMES, digits=3))

---
## Part 6 — Final Evaluation on Held-Out Test Set

This is the first time I touch the test set. Every decision before this point — which features to use, which model architecture to choose, which hyperparameters to tune — was made purely on the validation set. That means the numbers below are a genuine estimate of real-world performance, not a figure inflated by iterating on test data.

I evaluate the best model from Part 5, look at where errors concentrate, and check whether the model's confidence tracks its actual accuracy.

In [ ]:
model = xgb.XGBClassifier()
model.load_model(os.path.join(MODEL_DIR, 'xgb_sbert_only.json'))
print('Model loaded: xgb_sbert_only.json')

emb_test = np.load(os.path.join(EMBED_DIR, 'sbert_test.npy'))
emb_val  = np.load(os.path.join(EMBED_DIR, 'sbert_val.npy'))
print(f'Test embeddings : {emb_test.shape}')

df_test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
df_val  = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
y_test  = df_test['label'].values
y_val   = df_val['label'].values

with open(os.path.join(DATA_DIR, 'label_mapping.json')) as f:
    mapping  = json.load(f)
    id2label = {int(k): v for k, v in mapping['id2label'].items()}

CLASS_NAMES = [id2label[i] for i in sorted(id2label)]
N_CLASSES   = len(CLASS_NAMES)
print(f'Classes ({N_CLASSES}): {CLASS_NAMES}')

In [ ]:
X_test_sp = sp.csr_matrix(emb_test.astype(np.float32))
X_val_sp  = sp.csr_matrix(emb_val.astype(np.float32))

y_pred_test  = model.predict(X_test_sp)
y_proba_test = model.predict_proba(X_test_sp)
y_pred_val   = model.predict(X_val_sp)

print(f'Test predictions shape  : {y_pred_test.shape}')
print(f'Unique predicted classes: {sorted(np.unique(y_pred_test))}')

### Final test set metrics

In [ ]:
acc_test      = accuracy_score(y_test, y_pred_test)
macro_f1_test = f1_score(y_test, y_pred_test, average='macro')
macro_p_test  = precision_score(y_test, y_pred_test, average='macro')
macro_r_test  = recall_score(y_test, y_pred_test, average='macro')

acc_val      = accuracy_score(y_val, y_pred_val)
macro_f1_val = f1_score(y_val, y_pred_val, average='macro')

print('='*52)
print('  FINAL TEST SET RESULTS')
print('='*52)
print(f'  Accuracy         : {acc_test:.4f}  ({acc_test*100:.2f}%)')
print(f'  Macro F1         : {macro_f1_test:.4f}  ({macro_f1_test*100:.2f}%)')
print(f'  Macro Precision  : {macro_p_test:.4f}')
print(f'  Macro Recall     : {macro_r_test:.4f}')
print('='*52)

gap_f1 = abs(macro_f1_val - macro_f1_test)
print(f'\nVal  Macro F1 : {macro_f1_val:.4f}')
print(f'Test Macro F1 : {macro_f1_test:.4f}')
print(f'Val→Test gap  : {gap_f1:.4f}  (< 0.02 = no overfitting)')

In [ ]:
print('Per-class report (TEST SET):\n')
print(classification_report(y_test, y_pred_test, target_names=CLASS_NAMES, digits=3))

### Confusion matrix — final

In [ ]:
cm      = confusion_matrix(y_test, y_pred_test)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, vmin=0, vmax=1, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual',    fontsize=12)
ax.set_title('SBERT + XGBoost — Final Confusion Matrix (Test Set)', fontsize=13)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'final_confusion_matrix.png'), dpi=120)
plt.show()

### Error analysis

For each class I find the most common misclassification. This tells me which product boundaries are semantically blurry — for example, Credit Reporting and Debt Collection often use similar language about credit accounts and disputes.

In [ ]:
print('Error analysis — where does each class leak?\n')
print(f'{"True Class":<28} {"Correct":>8} {"Top mistake → predicted as":<35} {"Mistake %":>10}')
print('-' * 85)

for i, true_name in enumerate(CLASS_NAMES):
    row          = cm[i]
    total        = row.sum()
    wrong_counts = row.copy()
    wrong_counts[i] = 0
    top_idx   = wrong_counts.argmax()
    top_count = wrong_counts[top_idx]

    if top_count > 0:
        print(f'{true_name:<28} {cm[i,i]/total:>7.1%}   → {CLASS_NAMES[top_idx]:<35} {top_count/total:>9.1%}')
    else:
        print(f'{true_name:<28} {cm[i,i]/total:>7.1%}   → (no errors)')

### Confidence distribution

A well-calibrated model should be more confident when it's right and less confident when it's wrong. I plot the distribution of max predicted probability for correct vs incorrect predictions.

In [ ]:
max_proba  = y_proba_test.max(axis=1)
is_correct = (y_pred_test == y_test)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(max_proba[is_correct],  bins=40, alpha=0.7, color='steelblue',
        label=f'Correct  (n={is_correct.sum():,})')
ax.hist(max_proba[~is_correct], bins=40, alpha=0.7, color='coral',
        label=f'Wrong    (n={(~is_correct).sum():,})')
ax.set_xlabel('Max predicted probability (confidence)')
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution — Correct vs Wrong Predictions (Test Set)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confidence_distribution.png'), dpi=120)
plt.show()

print(f'Correct predictions — mean confidence: {max_proba[is_correct].mean():.3f}')
print(f'Wrong predictions   — mean confidence: {max_proba[~is_correct].mean():.3f}')

### Sample predictions

I look at 2 correct and 2 wrong predictions to qualitatively check whether the errors make intuitive sense.

In [ ]:
df_test['predicted'] = [id2label[p] for p in y_pred_test]
df_test['actual']    = [id2label[a] for a in y_test]
df_test['correct']   = df_test['predicted'] == df_test['actual']
df_test['confidence']= max_proba

correct_samples = df_test[df_test['correct']].sample(2, random_state=1)
wrong_samples   = df_test[~df_test['correct']].sample(2, random_state=1)
samples = pd.concat([correct_samples, wrong_samples])

for _, row in samples.iterrows():
    status = 'CORRECT' if row['correct'] else 'WRONG'
    print(f'{status}  |  Actual: {row["actual"]}  |  Predicted: {row["predicted"]}  |  Confidence: {row["confidence"]:.3f}')
    print(f'  Text: {str(row["narrative_clean"])[:200]}...')
    print()

### Project results summary

In [ ]:
f1_per_test   = f1_score(y_test, y_pred_test, average=None)
tfidf_f1_val  = [0.382, 0.457, 0.851, 0.150, 0.369, 0.345, 0.343]
tfidf_acc_val = 0.7118
tfidf_mf1_val = 0.4140

print('=' * 68)
print('  PROJECT RESULTS SUMMARY')
print('=' * 68)
print(f'{"":<30} {"TF-IDF (val)":>14} {"SBERT (test)":>14}')
print('-' * 68)
for i, name in enumerate(CLASS_NAMES):
    delta = f1_per_test[i] - tfidf_f1_val[i]
    arrow = "up" if delta > 0.01 else ("down" if delta < -0.01 else "same")
    print(f'  {name:<28} {tfidf_f1_val[i]:>14.3f} {f1_per_test[i]:>14.3f}  [{arrow} {delta:+.3f}]')
print('-' * 68)
print(f'  {"Macro F1":<28} {tfidf_mf1_val:>14.3f} {macro_f1_test:>14.3f}  [{macro_f1_test-tfidf_mf1_val:+.3f}]')
print(f'  {"Accuracy":<28} {tfidf_acc_val:>14.3f} {acc_test:>14.3f}  [{acc_test-tfidf_acc_val:+.3f}]')
print('=' * 68)
print(f'\n  Dataset : CFPB Consumer Complaint Database')
print(f'  Classes : {N_CLASSES} financial product categories')
print(f'  Train   : 139,771 complaints')
print(f'  Test    : {len(df_test):,} complaints (held-out, never seen during training)')